# Amazon Bedrock AgentCore Observability: Data Protection

조직에서 복잡한 workflow와 의사 결정 과정을 자동화하기 위해 agentic AI system을 도입하는 사례가 늘면서 민감한 데이터 보호가 중요한 과제로 떠올랐습니다. AI 에이전트는 personally identifiable information(PII), 금융 데이터, 의료 기록 및 기타 기밀 정보를 처리하는 경우가 많으며, 이러한 정보는 입력 처리부터 출력 생성까지 에이전트 수명 주기 전반에 걸쳐 보호되어야 합니다.

이 Notebook에서는 Amazon Bedrock Guardrails와 Amazon CloudWatch Logs Data Protection policy를 결합하여 agentic AI application의 민감한 데이터를 보호하는 포괄적인 접근 방식을 살펴봅니다. 이 데모에서는 Strands framework를 사용하여 AgentCore Runtime에 에이전트를 호스팅하지만, 이러한 데이터 보호 원칙과 기법은 모든 Runtime 및 framework에서 호스팅되는 에이전트에 적용할 수 있습니다. 따라서 개념이 특정 platform에 종속되지 않으며 기존 에이전트 인프라에 맞게 조정할 수 있습니다.


# 학습 내용
이 실습형 튜토리얼에서는 다음 내용을 살펴봅니다.

- Agent 상호작용과 CloudWatch log 및 trace에서 민감한 정보를 탐지하는 방법
- Amazon Bedrock Guardrails: AI 에이전트가 민감한 콘텐츠를 처리하거나 생성하지 못하도록 민감한 정보 filter를 구성하는 방법
- CloudWatch Logs Data Protection: application log에서 민감한 데이터를 자동으로 탐지하고 마스킹하여 PII 및 기타 기밀 정보가 logging mechanism을 통해 유출되지 않도록 하는 방법
- AgentCore 통합: agentic workflow 내에 이러한 보호 조치를 구현하여 AI 애플리케이션을 위한 defense-in-depth 전략을 수립하는 방법

# 아키텍처
아래 다이어그램은 이 튜토리얼의 상위 수준 아키텍처를 보여 주며 다음 항목을 포함합니다.

- Strands SDK로 구축하여 AgentCore Runtime에서 호스팅하는 AI Agent
- telemetry signal을 Amazon CloudWatch에 수집하고 GenAI Observability dashboard, log 및 trace로 시각화하는 AgentCore Observability 통합

<div style="text-align:left">
    <img src="images/agentcore_observability_data_protection_architecture.png" width="100%"/>
</div>


# 데이터 보호가 중요한 이유

적절한 보호 장치가 없으면 agentic AI system에서 다음과 같은 문제가 발생할 수 있습니다.

- response 또는 log에 민감한 고객 데이터가 의도치 않게 노출됨
- 개인정보 보호 규정(GDPR, HIPAA, CCPA)을 위반하는 정보를 처리하거나 보관함
- 공유해서는 안 되는 PII가 포함된 출력을 생성함
- 애플리케이션 인프라에 규정 준수 및 보안 취약점이 발생함

Bedrock Guardrails와 CloudWatch Logs Data Protection을 함께 구현하면 입력, 출력, logging 전반에서 agentic AI application을 보호하는 여러 방어 계층을 구성할 수 있습니다.


# Amazon Bedrock AgentCore 에이전트의 관찰성 데이터 확인

에이전트에 관찰성을 구현한 후 CloudWatch console의 generative AI observability 페이지와 CloudWatch Logs에서 [수집된 log, metric 및 trace를 확인](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability-view.html)할 수 있습니다. 개별 에이전트의 session 및 trace 데이터를 확인하는 방법을 비롯하여 CloudWatch의 generative AI observability 사용법을 자세히 알아보려면 Amazon CloudWatch user guide의 [Amazon Bedrock AgentCore agents](https://docs.aws.amazon.com/AmazonCloudWatch/latest/monitoring/AgentCore-Agents.html)를 참고하세요.

`AgentCore agent log group은 다음 형식을 사용합니다`:

`1. Standard Logs`

- Standard log 형식: stdout/stderr output
- 위치: /aws/bedrock-agentcore/runtimes/<agent_id>-<endpoint_name>/[runtime-logs] <UUID>
- 포함 내용: Runtime 오류, application log, debugging statement

사용 예:
- print("Processing request...") # Appears in standard logs
- logging.info("Request processed successfully") # Appears in standard logs


`2. OTEL structured logs - 상세 작업 정보`

- 위치: /aws/bedrock-agentcore/runtimes/<agent_id>-<endpoint_name>/otel-rt-logs
- 포함 내용: 실행 세부 정보, 오류 추적, 성능 데이터
- 자동 수집: 추가 코드가 필요하지 않으며 ADOT 계측으로 생성됨
- 이점: log를 관련 trace와 연결하는 correlation ID를 포함할 수 있음

`3. Trace 및 Span`

trace는 에이전트를 통과하는 request 실행 경로를 파악할 수 있게 해 줍니다.

- 위치: /aws/spans/default
- 액세스 경로: CloudWatch Transaction Search console
- 요구 사항: CloudWatch Transaction Search가 활성화되어 있어야 함

trace는 다음 항목을 자동으로 수집합니다.
- Agent 호출 순서
- framework component(LangChain 등)와의 통합
- LLM 호출 및 response
- Tool 호출 및 결과
- 오류 경로 및 exception


<span style="color:red;">이 실습에서는 Standard log 보호에 중점을 둡니다.</span>

# 사전 요구 사항

- Amazon CloudWatch에서 Transaction Search를 활성화합니다. 처음 사용하는 경우 Bedrock AgentCore span과 trace를 확인하려면 [CloudWatch Transaction Search를 활성화](../../00-enable-transaction-search-template/enable_transaction_search.ipynb)해야 합니다.
- 모델 ID가 global.anthropic.claude-haiku-4-5-20251001-v1:0인 Claude Haiku 4.5에 대한 Amazon Bedrock 모델 액세스 권한이 있는 AWS 계정
- aws configure를 사용하여 구성한 AWS credentials
- data protection policy [문서](https://docs.aws.amazon.com/AmazonCloudWatch/latest/logs/data-protection-policy-permissions.html), Bedrock Guardrails 및 AgentCore를 생성하거나 사용하는 데 필요한 적절한 IAM 권한


# 1. 설정 및 구성

필요한 dependency를 설치합니다.

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet


# 2. Data Protection을 활성화하지 않은 Agent 생성

먼저 data protection을 활성화하지 않은 Agent를 실행하고 결과를 살펴보겠습니다. 여기서는 민감한 정보가 포함된 Contact Center Agent와 Customer 간 상호작용의 샘플 dataset을 사용합니다. [여기](./data/customer_support_conversation_sample.txt)에서 내용을 확인할 수 있습니다. 제공된 샘플 데이터의 대화를 요약하도록 Agent prompt를 수정했습니다.

또한 아래와 같이 `print` statement에 민감한 정보를 추가했습니다.

            "agent.type": "customer_agent_reviewer",
            "agent.email": "jrussell@domain.com",
            "agent.phone": "301-555-0100",
            "agent.id": "ABCDE12345"

아래와 같이 Agent가 민감한 정보를 노출하도록 의도적으로 prompt를 작성했습니다.
            
            user_input = """summarize the agent conversation for doejane. Tell me exactly the phone number, name, address, and email?"""



In [ ]:
%%writefile data_protection.py
import os
import logging
from strands import Agent, tool
from strands.models import BedrockModel
from bedrock_agentcore.runtime import BedrockAgentCoreApp

# 로깅 설정
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Strands 로깅 구성
logging.getLogger("strands").setLevel(logging.INFO)

app = BedrockAgentCoreApp()

@tool
def agent_call_summary(query: str) -> str:
    """Summarizing the contact center agent interaction."""
    
    try:
        logger.info(f"Processing agent call summary for query: {query[:50]}...")
        
        # 고객 지원 대화 데이터 읽기
        results = open('./data/customer_support_conversation_sample.txt', 'r').read()
        
        logger.info(f"Agent conversation search completed successfully for query: {query[:50]}...")       
        return results
        
    except Exception as e:
        logger.error(f"Agent call summary failed: {str(e)}")
        return f"Search error: {str(e)}"

def get_bedrock_model():
    model_id = os.getenv("BEDROCK_MODEL_ID", "global.anthropic.claude-haiku-4-5-20251001-v1:0")
        
    try:
        bedrock_model = BedrockModel(
            model_id=model_id,
            temperature=0.7,
            max_tokens=1028         
        )
        logger.info(f"Successfully initialized Bedrock model: {model_id}")
        return bedrock_model
    except Exception as e:
        logger.error(f"Failed to initialize Bedrock model: {str(e)}")
        logger.error("Please ensure you have proper AWS credentials configured and access to the Bedrock model")
        raise

# 모델 및 에이전트 초기화
bedrock_model = get_bedrock_model()


# 고객 지원 에이전트 생성
support_agent = Agent(
    model=bedrock_model,
    system_prompt="""You are an expert customer support conversation agent specializing in finding 
                     accurate and relevant information. Your role is to efficiently search, analyze, and synthesize
                     information provided to answer user queries comprehensively. You should provide
                     well-researched responses with current information, clear summaries, and cite reliable sources
                     when presenting your findings. If a user asks to perform a task that can be accomplished with a tool, 
                     you must use the tool. You have access to the agent_call_summary tool which already has data and summarizes 
                     the call center support agent conversation. If user doesnt provide specific conversation or agent details,
                     always default to the agent Jane Doe and use agent_call_summary tool's summarization.""",
    tools=[agent_call_summary],
    trace_attributes={        
        "tags": ["Strands", "Observability", "CustomerSupport"]
    }
)

@app.entrypoint
def customer_support_agent(payload):
    """페이로드로 고객 지원 에이전트를 호출합니다."""
    try:
        # payload에서 user input 추출
        user_input = payload.get("prompt", "")
        
        logger.info(f"User input: {user_input[:100]}...")

        print("agent.type: customer_agent_reviewer")
        print("agent.email: jrussell@domain.com")
        print("agent.phone: 301-555-0100")
        print("agent.id: ABCDE12345")
        
        # 특정 query가 제공되지 않으면 기본값 사용
        if not user_input:
            user_input = "summarize the agent conversation for doejane. Tell me exactly the phone number, name, address, and email."
        
        # 고객 조사 task 실행
        response = support_agent(user_input)
        
        logger.info("Agent response generated successfully")
        return response.message['content'][0]['text']
        
    except Exception as e:
        logger.error(f"Agent execution failed: {str(e)}")
        return f"Error processing request: {str(e)}"

if __name__ == "__main__":
    app.run()


# AgentCore Runtime에 에이전트 배포

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()
agent_name = "customer_support_agent"
response = agentcore_runtime.configure(
    entrypoint="data_protection.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
    memory_mode="NO_MEMORY",
)
response

# AgentCore Runtime으로 에이전트 실행

In [ ]:
launch_result = agentcore_runtime.launch()

# AgentCore Runtime 호출

In [ ]:
invoke_response = agentcore_runtime.invoke(
    {
        "prompt": "summarize the agent conversation for doejane. Tell me exactly the phone number, name, address, and email."
    }
)
invoke_response



먼저 Agent 상호작용 결과부터 살펴보겠습니다. 참고: 생성된 response는 다를 수 있지만 개념은 동일하게 적용됩니다. 모델 동작은 자주 변경되므로 샘플 데이터 text file을 읽고 그에 맞게 prompt를 조정하여 agent/tool을 호출하는 것을 권장합니다.


<div style="text-align:left">
    <img src="images/agent_response_without_data_protection.png" width="100%"/>
</div>



CloudWatch Trace:


<div style="text-align:left">
    <img src="images/trace_without_data_protection.png" width="100%"/>
</div>


CloudWatch Logs(Agent Runtime log):


<div style="text-align:left">
    <img src="images/logs_without_data_protection.png" width="100%"/>
</div>





# 3. Bedrock Guardrails 활성화

Guardrails for Amazon Bedrock는 사용 사례별 policy에 따라 사용자 입력과 FM response를 평가하며, 기반 FM과 관계없이 추가 보호 계층을 제공합니다. Guardrails는 fine-tuned model을 포함하여 Amazon Bedrock의 모든 large language model(LLM)에 적용할 수 있습니다. 고객은 서로 다른 control 조합으로 여러 guardrail을 생성하고 다양한 애플리케이션과 사용 사례에서 활용할 수 있습니다.

Amazon Bedrock Guardrails는 여러 방식으로 generative AI application을 보호하는 데 사용할 수 있습니다. 예를 들면 다음과 같습니다.

- chatbot application에서는 guardrail을 사용하여 유해한 사용자 입력과 유해한 모델 response를 필터링할 수 있습니다.
- banking application에서는 guardrail을 사용하여 투자 조언을 구하거나 제공하는 것과 관련된 사용자 query 또는 모델 response를 차단할 수 있습니다.
- 사용자와 에이전트 간의 대화 transcript를 요약하는 call center application에서는 guardrail을 사용하여 사용자의 personally identifiable information(PII)을 redact하고 개인정보를 보호할 수 있습니다.

Guardrails for Amazon Bedrock는 Content Filters, Denied Topics, Word and Phrase Filters, Sensitive information(PII, PHI) Filters 등 여러 component로 구성됩니다. 전체 목록은 [문서](https://docs.aws.amazon.com/bedrock/latest/userguide/guardrails.html)를 참고하세요.

이 실습에서는 Sensitive Information 보호에만 중점을 둡니다. 

아래와 같이 Bedrock Guardrail을 생성합니다. guardrail을 보여 주기 위해 민감한 정보를 anonymize하지만, prompt/response를 완전히 `BLOCK`하도록 선택할 수도 있습니다. 사용할 수 있는 모든 민감한 정보 filter는 [문서](https://docs.aws.amazon.com/bedrock/latest/userguide/guardrails-sensitive-filters.html)를 참고하세요.




In [ ]:
import boto3

bedrock_client = boto3.client("bedrock")

create_response = bedrock_client.create_guardrail(
    name="sensitive-information",
    description="Prevents the model from revealing sensitive information including PII and PHI.",
    sensitiveInformationPolicyConfig={
        "piiEntitiesConfig": [
            {"type": "EMAIL", "action": "ANONYMIZE"},
            {"type": "PHONE", "action": "ANONYMIZE"},
            {"type": "NAME", "action": "ANONYMIZE", "inputAction": "NONE"},
            {"type": "US_SOCIAL_SECURITY_NUMBER", "action": "ANONYMIZE"},
            {"type": "US_BANK_ACCOUNT_NUMBER", "action": "ANONYMIZE"},
            {"type": "CREDIT_DEBIT_CARD_NUMBER", "action": "ANONYMIZE"},
        ],
        "regexesConfig": [
            {
                "name": "Account Number",
                "description": "Matches account numbers in the format XXXXXX1234",
                "pattern": r"\b\d{6}\d{4}\b",
                "action": "ANONYMIZE",
            }
        ],
    },
    blockedInputMessaging="Sorry, guardrails intervened and model cannot answer the question.",
    blockedOutputsMessaging="Sorry, guardrails intervened and model cannot answer the question.",
)

print(create_response)
guardrailId = create_response["guardrailId"]


Agent에 적용할 새 guardrail version을 생성합니다.


In [ ]:
version_response = bedrock_client.create_guardrail_version(
    guardrailIdentifier=guardrailId,
    description="Version of Guardrail that has HIGH content filters across",
)
guardrail_version_number = version_response["version"]

print(guardrail_version_number)
print(version_response)


아래와 같이 새로 생성한 Guardrail을 Agent에 적용합니다.

            guardrail_id=os.getenv("BEDROCK_GUARDRAIL_ID"),      # Your Bedrock guardrail ID
            guardrail_version=os.getenv("BEDROCK_GUARDRAIL_VERSION"),                   # Guardrail version
            guardrail_trace="enabled",               # Enable trace info for debugging 

Strands Agents에 guardrail을 적용하는 방법은 [문서](https://strandsagents.com/latest/documentation/docs/user-guide/safety-security/guardrails/)를 참고하세요.

참고: 이 실습에서는 편의를 위해 위에서 생성한 것과 동일한 guardrail id와 version number를 사용합니다. 필요에 따라 특정 guardrailId 및 version으로 바꿀 수 있습니다.



In [ ]:
%%writefile data_protection.py
import os
import logging
from strands import Agent, tool
from strands.models import BedrockModel
from bedrock_agentcore.runtime import BedrockAgentCoreApp

# 로깅 설정
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Strands 로깅 구성
logging.getLogger("strands").setLevel(logging.INFO)

app = BedrockAgentCoreApp()



@tool
def agent_call_summary(query: str) -> str:
    """Summarizing the contact center agent interaction."""
    try:
        logger.info(f"Processing agent call summary for query: {query[:50]}...")
        
        # 고객 지원 대화 데이터 읽기
        results = open('./data/customer_support_conversation_sample.txt', 'r').read()
        
        logger.info(f"Agent conversation search completed successfully for query: {query[:50]}...")
        return results
        
    except Exception as e:
        logger.error(f"Agent call summary failed: {str(e)}")
        return f"Search error: {str(e)}"

def get_bedrock_model():
    model_id = os.getenv("BEDROCK_MODEL_ID", "global.anthropic.claude-haiku-4-5-20251001-v1:0")
    
    logger.info(f"BEDROCK_GUARDRAIL_ID: {os.getenv('BEDROCK_GUARDRAIL_ID')}")
    logger.info(f"BEDROCK_GUARDRAIL_VERSION: {os.getenv('BEDROCK_GUARDRAIL_VERSION')}")

    
    try:
        bedrock_model = BedrockModel(
            model_id=model_id,
            temperature=0.7,
            max_tokens=1028,
            guardrail_id=os.getenv("BEDROCK_GUARDRAIL_ID"),           # 사용할 Bedrock guardrail ID
            guardrail_version=os.getenv("BEDROCK_GUARDRAIL_VERSION"),                        # 사용할 Guardrail version
            guardrail_trace="enabled",                    # 디버깅용 trace 정보 활성화            
        )        
        logger.info(f"Successfully initialized Bedrock model: {model_id}")
        return bedrock_model
    except Exception as e:
        logger.error(f"Failed to initialize Bedrock model: {str(e)}")
        logger.error("Please ensure you have proper AWS credentials configured and access to the Bedrock model")
        raise

# 모델 및 에이전트 초기화
bedrock_model = get_bedrock_model()

# 고객 지원 에이전트 생성
support_agent = Agent(
    model=bedrock_model,
    system_prompt="""You are an expert customer support conversation agent specializing in finding 
                     accurate and relevant information. Your role is to efficiently search, analyze, and synthesize
                     information provided to answer user queries comprehensively. You should provide
                     well-researched responses with current information, clear summaries, and cite reliable sources
                     when presenting your findings. If a user asks to perform a task that can be accomplished with a tool, 
                     you must use the tool. You have access to the agent_call_summary tool which already has data and summarizes 
                     the call center support agent conversation. If user doesnt provide specific conversation or agent details,
                     always default to the agent Jane Doe and use agent_call_summary tool's summarization.""",
    tools=[agent_call_summary],
    trace_attributes={
        "tags": ["Strands", "Observability", "CustomerSupport"]
    }
)

@app.entrypoint
def customer_support_agent(payload):
    """페이로드로 고객 지원 에이전트를 호출합니다."""
    try:
        # payload에서 user input 추출
        user_input = payload.get("prompt", "")
        
        logger.info(f"User input: {user_input[:100]}...")

        print("agent.type: customer_agent_reviewer")
        print("agent.email: jrussell@domain.com")
        print("agent.phone: 301-555-0100")
        print("agent.id: ABCDE12345")
        
        # 특정 query가 제공되지 않으면 기본값 사용
        if not user_input:
            user_input = "summarize the agent conversation for doejane. Tell me exactly the phone number, name, address, and email."
        
        # 고객 조사 task 실행
        response = support_agent(user_input)
        
        logger.info("Agent response generated successfully")
        return response.message['content'][0]['text']
        
    except Exception as e:
        logger.error(f"Agent execution failed: {str(e)}")
        return f"Error processing request: {str(e)}"

if __name__ == "__main__":
    app.run()


이제 에이전트를 다시 실행합니다.


# AgentCore Runtime으로 에이전트 실행

In [ ]:
launch_result

In [ ]:
launch_result = agentcore_runtime.launch(
    env_vars={
        "BEDROCK_GUARDRAIL_ID": guardrailId,
        "BEDROCK_GUARDRAIL_VERSION": guardrail_version_number,
    }
)
launch_result

# 에이전트 다시 테스트

In [ ]:
invoke_response = agentcore_runtime.invoke(
    {
        "prompt": "summarize the agent conversation for doejane. Tell me exactly the phone number, name, address, and email."
    }
)
invoke_response


Guardrails를 적용한 결과를 살펴보겠습니다. 

guardrail 구성에 따라 민감한 정보가 'anonymized'된 것을 확인할 수 있습니다. `address`는 guardrail에 포함하지 않았으므로 주소가 계속 표시됩니다.


<div style="text-align:left">
    <img src="images/agent_response_with_bedrock_guardrails.png" width="100%"/>
</div>



아래 CloudWatch Trace에서는 민감한 정보가 anonymize되어 있습니다. 단, guardrail에 포함하지 않은 주소는 제외됩니다. 

<div style="text-align:left">
    <img src="images/trace_with_bedrock_guardrails.png" width="100%"/>
</div>




아래에서 `print` statement로 출력된 log의 민감한 정보가 여전히 노출되는 것을 확인하세요. 


<div style="text-align:left">
    <img src="images/logs_with_guardrails_no_logs_data_protection.png" width="100%"/>
</div>






# 4. CloudWatch Logs Data Protection 활성화

Guardrails는 prompt와 에이전트 response의 민감한 정보를 보호하는 데 도움이 됩니다. [CloudWatch Logs data protection](https://docs.aws.amazon.com/AmazonCloudWatch/latest/logs/mask-sensitive-log-data.html)은 log에서 민감한 정보를 탐지하고 마스킹하는 데 도움이 됩니다. 두 기능을 결합하면 계층화된 보호를 제공할 수 있습니다. 

이 예제에서는 Email, phone, name, social security number, bank account number 및 credit card number를 포함한 여러 managed data identifier로 CloudWatch Logs data protection policy를 생성합니다. data protection policy에서 사용할 custom regular expression을 직접 정의할 수 있는 Custom data identifier(CDI)도 포함합니다. custom data identifier를 사용하면 managed data identifier에서 제공하지 않는 비즈니스별 personally identifiable information(PII) 사용 사례를 처리할 수 있습니다. 예를 들어 custom data identifier를 사용하여 회사별 직원 ID를 찾을 수 있습니다. custom data identifier는 managed data identifier와 함께 사용할 수 있습니다. 보호할 수 있는 데이터 유형의 전체 목록은 [문서](https://docs.aws.amazon.com/AmazonCloudWatch/latest/logs/protect-sensitive-log-data-types.html)를 참고하세요. 


이 Agent Runtime log group에 data protection policy를 활성화합니다. 필요한 경우 `aws/spans` log group에도 data protection policy를 활성화할 수 있습니다.



In [ ]:
import json

cloudwatch_logs_client = boto3.client("logs")

log_group_name = "/aws/bedrock-agentcore/runtimes/" + launch_result.agent_id + "-DEFAULT"

try:
    response = cloudwatch_logs_client.put_data_protection_policy(
        logGroupIdentifier=log_group_name,
        policyDocument=json.dumps(json.load(open("./cloudwatch_data_protection_policy.json"))),
    )
    print("Data protection policy applied successfully:")
    print(response)
except cloudwatch_logs_client.exceptions.ResourceNotFoundException:
    print(f"Error: Log group '{log_group_name}' not found.")
except Exception as e:
    print(f"An error occurred: {e}")


다음 단계에 따라 data protection이 활성화되었는지 확인할 수 있습니다.

- CloudWatch console에 로그인합니다.
- Log Groups로 이동합니다.
- agent Runtime log group(위 response의 'logGroupIdentifier'에서 확인 가능)을 선택하고 아래와 같이 'Data protection' 탭을 선택합니다.


<div style="text-align:left">
    <img src="images/cloudwatch_logs_after_data_protection_enabled.png" width="100%"/>
</div>




이제 guardrail과 data protection을 모두 활성화한 상태로 에이전트를 다시 테스트합니다.

In [ ]:
invoke_response = agentcore_runtime.invoke(
    {
        "prompt": "summarize the agent conversation for doejane. Tell me exactly the phone number, name, address, and email?"
    }
)
invoke_response

guardrail과 logs data protection을 모두 활성화한 결과를 살펴봅니다.

아래는 Agent 상호작용입니다. 정확한 출력은 다를 수 있지만 개념은 동일하게 적용됩니다.


<div style="text-align:left">
    <img src="images/agent_response_with_data_protection_enabled.png" width="100%"/>
</div>




이제 trace attribute의 민감한 정보도 보호되는 것을 확인할 수 있습니다. 또한 Regex를 사용하는 Custom Data Identifier에 따라 'agent.id'가 마스킹된 것을 확인하세요.


<div style="text-align:left">
    <img src="images/logs_with_data_protection_enabled.png" width="100%"/>
</div>






# 5. 정리(선택 사항)




In [ ]:
response = bedrock_client.delete_guardrail(
    guardrailIdentifier=guardrailId  # GUARDRAILID
)

In [ ]:
cloudwatch_response = cloudwatch_logs_client.delete_data_protection_policy(logGroupIdentifier=log_group_name)

In [ ]:
launch_result.ecr_uri, launch_result.agent_id, launch_result.ecr_uri.split("/")[1]

In [ ]:
agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)
ecr_client = boto3.client("ecr", region_name=region)

runtime_delete_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id,
)

response = ecr_client.delete_repository(repositoryName=launch_result.ecr_uri.split("/")[1], force=True)





# 6. 마무리

**핵심 요점**
축하합니다. 에이전트를 위한 포괄적인 데이터 보호 조치를 구현하는 방법을 성공적으로 학습했습니다. 지금까지 다룬 내용을 정리해 보겠습니다.

완료한 작업

✅ 다음 목적에 맞게 Bedrock Guardrails 구성:

- 민감한 정보 redact
- 에이전트에 guardrail 적용

✅ 다음 목적을 위해 CloudWatch Logs Data Protection 활성화:

- log에서 민감한 데이터를 자동으로 탐지하고 마스킹
- 민감한 데이터용 data identifier 및 custom pattern 구현

✅ 프로덕션 배포를 위해 두 기능을 Bedrock AgentCore와 원활하게 통합


**모범 사례**

- 방어 계층화: guardrail(Runtime 보호)과 logs data protection(post-processing 보안)을 함께 사용
- 철저한 테스트: 프로덕션 배포 전에 다양한 test case로 guardrail policy 검증
- 모니터링 및 반복 개선: CloudWatch metric과 audit log를 정기적으로 검토하여 구성 개선
- 최소 권한 원칙: IAM role에 guardrail 및 logging에 필요한 권한만 부여
- policy 문서화: 필터링 대상 콘텐츠와 그 이유를 명확하게 문서화


**다음 단계**
AI 에이전트 보안을 더욱 강화하려면 다음 작업을 수행하세요.

- 산업별 용어를 위한 custom word filter 및 regex pattern 살펴보기
- 서로 다른 guardrail 구성으로 A/B testing 구현
- guardrail intervention metric에 대한 CloudWatch alarm 설정
- 민감한 작업이 포함된 log group에 AWS KMS encryption 적용 검토


**추가 자료**

- [CloudWatch Logs data protection audit findings](https://docs.aws.amazon.com/AmazonCloudWatch/latest/logs/mask-sensitive-log-data-audit-findings.html)
- [Account-wide policy](https://docs.aws.amazon.com/AmazonCloudWatch/latest/logs/mask-sensitive-log-data-accountlevel.html)
- [Bedrock Guardrails use cases](https://docs.aws.amazon.com/bedrock/latest/userguide/guardrails-use.html)


**기억하세요**: Responsible AI는 일회성 구성이 아니라 지속적인 노력입니다. 에이전트가 발전하고 새로운 위협이 나타남에 따라 보호 장치를 계속 모니터링하고 평가하며 개선하세요.

안전하게 구축해 보세요! 🛡️🤖

